## In-Class Lecture 34-1 - Making a Command Queue Flight-Ready

In this lecture, you will take a working command queue, the kind every spacecraft uses to store commands from the ground, and bring it into compliance with Power of Ten Rules 1, 2, and 3. Each rule you apply fixes a failure you will see happen first.

### Reviewing Opening your Environment on the Notre Dame Machines

<b>1.</b> Log into the Notre Dame Machines as you performed on Homework 1 using Tabby Terminal 

Open Tabby Terminal on your machine.

Type in: <code>ssh -Y username@student13.cse.nd.edu</code>
<ul><li>"username" is your NetID.</li>
<ul>
<li>Example: my email is matt.morrison@nd.edu, but my NetID is mmorri22, so I would enter:<p>
<code>ssh -Y mmorri22@student13.cse.nd.edu</code><p>
The <code>-Y</code> is included to treat the Notre Dame machines as a trusted client<p>
</li>
</ul>
</ul>

2. Next, we will create a directory where you will save your In-Class Code. First, we will go into the Dropbox directory where you will submit your code:

<code>cd esc-courses</code>

<code>cd fa26-cse-20133.01</code>

<code>cd dropbox</code>

Make a directory for you to practice your code for this lecture

    mkdir inclass34
    cd inclass34

Enable script so you can run the most up-to-date versions of C and C++:

    scl enable gcc-toolset-15 bash

> <b>Note:</b> If you use the <code>-std=c2x</code> and get the following error, this means you did not run the enable command<br>
> <code>gcc: error: unrecognized command line option ‘-std=c2x’; did you mean ‘-std=c1x’?</code>

To obtain the code for today's lecture, perform the following commands:

    wget https://raw.githubusercontent.com/mmorri22/cse20133/main/inclass/lec34/setup.sh
    chmod a+rx setup.sh
    ./setup.sh
    
To confirm, perform the <code>ls</code> command. You will see the following:

    burst.txt  cmd_queue.c  commands.txt  Makefile

### In-Class Reading Discussion

<font color = "red">Class Introduction Question #1 - What is the difference between a MISRA <i>required</i> rule and an <i>advisory</i> rule, and what must a team do to break a required rule?</font>

<font color = "red">Class Introduction Question #2 - Pick one bug from Lecture 32. Name the CERT rule it violates, and explain in one sentence what the compliant version of the code would do differently.</font>

<font color = "red">Class Introduction Question #3 - Rule 1 forbids recursion. Why does recursion make it impossible for a tool to prove the stack is large enough, and why did <code>recursive_delete</code> fail at 5,000 nodes with a 64 KB stack but not with 8 MB?</font>

<font color = "red">Class Introduction Question #4 - <code>find_unbounded</code> works correctly whenever the key is in the list. Describe the input that makes it fail, and explain how the bounded version turns a hang into a reported anomaly.</font>

<font color = "red">Class Introduction Question #5 - Give two reasons flight software avoids <code>malloc</code> after initialization. What happens in <code>node_pool.c</code> when the pool runs out, and why is that better than <code>malloc</code> returning <code>NULL</code> in the middle of a mission?</font>

### The Program

<code>cmd_queue.c</code> reads commands from a file, one per line: an ID, the time to execute it, and an opcode. It loads them into a queue (a singly linked list with <code>head</code> and <code>tail</code> pointers), cancels one command by ID, and then executes the rest in order.

    ./cmd_queue <command_file> <id_to_cancel> [fault]

The spacecraft has room for <code>MAX_COMMANDS</code> (16) commands. The optional word <code>fault</code> simulates a radiation bit flip that corrupts one <code>next</code> pointer.

### Step 0 - Watch It Fail

Run all three tests before you change anything:

    make cmd_queue
    make burst
    make fault

<ul>
    <li><code>make cmd_queue</code> works perfectly.</li>
    <li><code>make burst</code> sends 20 commands. How many did the spacecraft accept? Is that possible on a computer with room for 16?</li>
    <li><code>make fault</code> never finishes. <code>timeout</code>, standing in for the watchdog, kills it after 2 seconds with exit status 124.</li>
</ul>

Also run <code>make check_heap</code>. It searches the code for heap functions and fails if it finds any. Which lines does it report?

### Step 1 - Rule 1: Replace the Recursion

Find the three recursive functions: <code>count_commands</code>, <code>print_commands</code>, and <code>delete_commands</code>. Rewrite each one with a loop, using <code>destructor_iterative</code> from Reading 34-2 as your model.

Remember to save the <code>next</code> pointer <i>before</i> you release a command in <code>delete_commands</code>.

Run <code>make cmd_queue</code>. The output should be exactly the same as in Step 0.

### Step 2 - Rule 2: Bound Every Loop

Every loop in the program walks the queue. A correct queue can never hold more than <code>MAX_COMMANDS</code> commands, so no loop should ever take more than <code>MAX_COMMANDS</code> steps.
<ul>
    <li>Add a bound to <code>print_commands</code>, <code>execute_all</code>, and the loop you wrote in <code>delete_commands</code>.</li>
    <li>In <code>count_commands</code>, return <code>-1</code> if the count goes past <code>MAX_COMMANDS</code>. That means the queue is corrupted.</li>
    <li>In <code>cancel_command</code>, return <code>-2</code> if the search takes more than <code>MAX_COMMANDS</code> steps.</li>
</ul>

Now decide what the spacecraft should do with a corrupted queue. In <code>main</code>, after the cancel: if <code>cancel_command</code> returned <code>-2</code> or <code>count_commands</code> returns <code>-1</code>, print an anomaly message, announce that the spacecraft is entering safe mode, and return <code>EXIT_FAILURE</code> without executing anything.

    make fault

It should now finish immediately with your safe-mode message. <code>make</code> will report <code>Error 1</code>, because your program correctly returned <code>EXIT_FAILURE</code>. Compare that to <code>Error 124</code> from Step 0. What is the difference between the two?

### Step 3 - Rule 3: Replace <code>malloc</code> with a Pool

Using <code>node_pool.c</code> from Reading 34-2 as your model:
<ol>
    <li>Add a static array of <code>MAX_COMMANDS</code> commands and a static <code>free_list</code> pointer.</li>
    <li>Write <code>pool_init</code>, and call it once at the start of <code>main</code>.</li>
    <li>Rewrite <code>cmd_alloc</code> and <code>cmd_release</code> to use the free list instead of <code>malloc</code> and <code>free</code>.</li>
</ol>

Then run:

    make check_heap
    make burst

<code>check_heap</code> should now pass, and <code>burst</code> should reject exactly 4 commands.

<b>Think about it:</b> Do you still need <code>delete_commands</code>? Where does every command's memory end up after <code>execute_all</code> finishes? What would your safe-mode code need to do to make the whole pool usable again?

### Discussion

<ol>
    <li>In safe mode, the spacecraft discards every stored command. Is that the right choice? What would the ground team need to do next, and what could go wrong if the spacecraft tried to execute part of a corrupted queue instead?</li>
    <li>We chose 16 for <code>MAX_COMMANDS</code>. What would you need to know about the mission to justify that number?</li>
    <li><code>make check_heap</code> verifies Rule 3 automatically with a simple text search. Could a text search verify Rule 1 or Rule 2? What would a tool need to understand about the code to check them?</li>
</ol>

## Compilation Run

    make cmd_queue
    make burst
    make fault
    make analyze
    make check_heap

When you are finished, <code>cmd_queue</code> and <code>burst</code> run to completion, <code>fault</code> enters safe mode immediately, <code>analyze</code> reports no errors, and <code>check_heap</code> passes.